# Biodiversity 

In [1]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import fiona
import os

### load data

In [2]:
data_dir = os.path.join(os.getcwd(), 'data')

# Path to the geodatabase
gdb_path = os.path.join(data_dir, 'Lebensraumkarte_v1_1_VD_20241025.gdb')

# List all layers in the geodatabase
layers = fiona.listlayers(gdb_path)

# Read a specific layer
gdf = gpd.read_file(gdb_path, layer=layers[0])

bike_path = os.path.join(data_dir, 'lausanne_bike_edges.gpkg')
bike_roads = gpd.read_file(bike_path)

drive_path = os.path.join(data_dir, 'lausanne_drive_edges.gpkg')
drive_roads = gpd.read_file(drive_path)

walk_path = os.path.join(data_dir, 'lausanne_walk_edges.gpkg')
walk_roads = gpd.read_file(walk_path)

/home/quentin/Documents/bluecity-viz/.venv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  return ogr_read(


### Regroup all road types

In [3]:
bike_roads["type"] = "bike"
drive_roads["type"] = "drive"
walk_roads["type"] = "walk"

roads = pd.concat([bike_roads, drive_roads, walk_roads], ignore_index=True)

### Select biodiversity areas within Lausanne

In [8]:
# select only biodiversity areas within Lausanne
lausanne_boundary = gpd.read_file(os.path.join(data_dir, 'Lausanne Districts.gpkg'))
lausanne_boundary["geometry"] = lausanne_boundary.geometry.buffer(10)  # Fix potential geometry issues
gdf_lausanne = gpd.sjoin(gdf, lausanne_boundary, how='inner', predicate='intersects')
gdf_lausanne.drop(columns=['index_right'], inplace=True)

### Find biodiversity areas within a certain buffer distance of roads

In [ ]:
# For each road, find all the biodiversity areas within a 10m buffer
buffer_distance = 10  # meters

# Create a buffer around each road
roads.crs =  gdf_lausanne.crs
roads['geometry'] = roads.geometry.buffer(buffer_distance)

# Spatial join to find biodiversity areas within each buffered road
roads_with_biodiversity = gpd.sjoin(roads, gdf_lausanne, how='left', predicate='intersects')
roads_with_biodiversity = roads_with_biodiversity.drop(columns=['index_right'])
roads_with_biodiversity['cluster'] = None

In [18]:
roads_with_biodiversity.columns

Index(['u', 'v', 'key', 'osmid', 'highway', 'lanes', 'maxspeed', 'name', 'ref',
       'oneway', 'reversed', 'length', 'service', 'access', 'bridge',
       'junction', 'width', 'tunnel', 'geometry', 'type', 'index_right',
       'Cover', 'TypoCH_NUM', 'POLYID', 'Version', 'TypoCH', 'Probability',
       'Shape_Length', 'Shape_Area', 'Entity', 'Handle', 'Layer', 'LyrFrzn',
       'LyrOn', 'Color', 'Linetype', 'Elevation', 'LineWt', 'RefName',
       'DocUpdate', 'DocId', 'DI', 'PC', 'DV', 'Ver', 'cluster'],
      dtype='str')